# Advanced Feature Engineering Cookbook: 50 Techniques That Win Competitions

**A comprehensive, practical guide to feature engineering for tabular data**

---

> **TL;DR** -- This cookbook implements **50 feature engineering techniques** organized into 8 categories: numerical transforms, categorical encoding, datetime features, text features, interactions, aggregations, and target-based features. Each technique includes working code, an explanation of when to use it, and a **measured ROC-AUC impact**. We show the cumulative improvement from baseline to full pipeline on a synthetic customer churn dataset. Everything runs on Kaggle with zero external dependencies.

### **Key Results at a Glance**
| Category | Techniques | Example Impact |
|:---|:---|:---|
| Numerical Transforms | Log, Power, Scale, Bin, Clip, Rank | Foundation for all models |
| Categorical Encoding | One-Hot, Frequency, Target, WoE, Hash | Unlock categorical signal |
| DateTime Features | Cyclical, Time-Since, Seasonality | Free signal most people miss |
| Text Features | TF-IDF, Sentiment, Char n-grams | Extract structure from text |
| Interactions | Multiply, Ratio, Difference, Cat x Num | Capture relationships |
| Aggregations | Group stats, Z-scores, Rank-in-group | Context-aware features |
| Target-Based | K-Fold Target Encoding, Binned Rates | Most powerful (handle with care!) |
| **Full Pipeline** | **All combined** | **Measurable AUC improvement** |

### **Applicable Kaggle Competitions**
- [Tabular Playground Series](https://www.kaggle.com/competitions?search=tabular+playground) -- all editions
- [Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk) -- feature engineering is 90% of the solution
- [Enefit - Predict Energy Behavior](https://www.kaggle.com/competitions/predict-energy-behavior-of-prosumers) -- datetime + aggregation heavy
- [American Express Default Prediction](https://www.kaggle.com/competitions/amex-default-prediction) -- aggregation + target encoding
- Any **tabular classification or regression** competition

---

If you find this notebook helpful, please **upvote** -- it helps others find it too!

## 📑 Table of Contents

1. [🔧 Setup & Synthetic Data Generation](#1)
2. [📊 Numerical Transformations](#2) (Techniques 1-10)
3. [🏷️ Categorical Encoding](#3) (Techniques 11-20)
4. [📅 DateTime Features](#4) (Techniques 21-27)
5. [📝 Text Features](#5) (Techniques 28-34)
6. [🔗 Interaction Features](#6) (Techniques 35-40)
7. [📊 Aggregation Features](#7) (Techniques 41-45)
8. [🎯 Target-Based Features](#8) (Techniques 46-50)
9. [🚀 Full Pipeline: Combined Effect](#9)
10. [📝 Key Takeaways](#10)

<a id='1'></a>
## 1. 🔧 Setup & Synthetic Data Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, 
    LabelEncoder, OneHotEncoder, PolynomialFeatures,
    PowerTransformer, QuantileTransformer, KBinsDiscretizer
)
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('All imports successful!')

### Generate Synthetic Dataset

We create a realistic tabular dataset simulating customer churn prediction with numerical, categorical, datetime, and text features.

In [ ]:
def generate_synthetic_data(n=5000):
    """Generate a realistic synthetic dataset for feature engineering demos."""
    np.random.seed(42)
    
    # Numerical features
    age = np.random.normal(40, 15, n).clip(18, 80)
    income = np.random.lognormal(10.5, 0.8, n)  # skewed
    tenure_months = np.random.exponential(24, n).clip(1, 120)
    monthly_charges = np.random.uniform(20, 150, n)
    total_charges = monthly_charges * tenure_months * np.random.uniform(0.8, 1.2, n)
    num_products = np.random.poisson(2, n).clip(0, 8)
    support_calls = np.random.poisson(1.5, n)
    days_since_last_activity = np.random.exponential(30, n).clip(0, 365)
    satisfaction_score = np.random.uniform(1, 10, n)
    usage_minutes = np.random.lognormal(5, 1.5, n)
    
    # Categorical features
    gender = np.random.choice(['Male', 'Female', 'Other'], n, p=[0.48, 0.48, 0.04])
    region = np.random.choice(['North', 'South', 'East', 'West', 'Central'], n)
    plan_type = np.random.choice(['Basic', 'Standard', 'Premium', 'Enterprise'], n, 
                                  p=[0.4, 0.3, 0.2, 0.1])
    payment_method = np.random.choice(['Credit Card', 'Bank Transfer', 'PayPal', 'Cash'], n)
    device_type = np.random.choice(['Mobile', 'Desktop', 'Tablet'], n, p=[0.5, 0.35, 0.15])
    
    # High-cardinality categorical
    city = np.random.choice([f'City_{i}' for i in range(200)], n)
    
    # DateTime features
    signup_date = pd.to_datetime('2020-01-01') + pd.to_timedelta(
        np.random.randint(0, 1500, n), unit='D')
    last_login = signup_date + pd.to_timedelta(
        np.random.randint(0, 365, n), unit='D')
    
    # Text features
    feedback_words = [
        'great service love it fast reliable',
        'terrible slow bad experience disappointed',
        'okay average nothing special decent',
        'amazing wonderful excellent top notch',
        'poor quality worst ever regret purchase',
        'good value money reasonable price',
        'needs improvement could better sometimes',
        'fantastic support helpful team quick response',
        'frustrating complicated confusing interface',
        'satisfied overall happy recommend friends',
    ]
    feedback = np.array([feedback_words[np.random.randint(0, len(feedback_words))] for _ in range(n)])
    
    # Target: churn (binary) - correlated with features
    churn_prob = (
        0.3 
        - 0.003 * tenure_months 
        + 0.02 * support_calls 
        - 0.02 * satisfaction_score
        + 0.001 * days_since_last_activity
        + 0.1 * (plan_type == 'Basic').astype(float)
        - 0.1 * (plan_type == 'Enterprise').astype(float)
        + np.random.normal(0, 0.1, n)
    )
    churn_prob = np.clip(churn_prob, 0.05, 0.95)
    target = (np.random.random(n) < churn_prob).astype(int)
    
    df = pd.DataFrame({
        'age': age, 'income': income, 'tenure_months': tenure_months,
        'monthly_charges': monthly_charges, 'total_charges': total_charges,
        'num_products': num_products, 'support_calls': support_calls,
        'days_since_last_activity': days_since_last_activity,
        'satisfaction_score': satisfaction_score, 'usage_minutes': usage_minutes,
        'gender': gender, 'region': region, 'plan_type': plan_type,
        'payment_method': payment_method, 'device_type': device_type,
        'city': city, 'signup_date': signup_date, 'last_login': last_login,
        'feedback': feedback, 'target': target
    })
    
    # Add some missing values (realistic)
    for col in ['income', 'satisfaction_score', 'feedback']:
        mask = np.random.random(n) < 0.05
        df.loc[mask, col] = np.nan
    
    return df


df = generate_synthetic_data(5000)
print(f'Dataset shape: {df.shape}')
print(f'Target distribution:\n{df["target"].value_counts(normalize=True).round(3)}')
print(f'\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head()

### Evaluation Helper

We'll use this helper to measure the impact of each feature engineering technique.

In [ ]:
def evaluate_features(df, feature_cols, target_col='target', model=None, cv=5):
    """Evaluate features using cross-validated ROC-AUC."""
    X = df[feature_cols].copy()
    y = df[target_col]
    
    # Handle non-numeric columns
    for col in X.columns:
        if X[col].dtype == 'object':
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))
        elif X[col].dtype in ['datetime64[ns]', 'datetime64']:
            X[col] = X[col].astype(np.int64) // 10**9
    
    X = X.fillna(X.median())
    
    if model is None:
        model = GradientBoostingClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1,
            subsample=0.8, random_state=42
        )
    
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    return scores.mean(), scores.std()


# Baseline: use raw numerical features
num_cols = ['age', 'income', 'tenure_months', 'monthly_charges', 'total_charges',
            'num_products', 'support_calls', 'days_since_last_activity',
            'satisfaction_score', 'usage_minutes']

baseline_auc, baseline_std = evaluate_features(df, num_cols)
print(f'Baseline ROC-AUC: {baseline_auc:.4f} (+/- {baseline_std:.4f})')

# Track all improvements
improvements = {'Baseline': baseline_auc}

<a id='2'></a>
## 2. 📊 Numerical Transformations (Techniques 1-10)

### Technique 1: Log Transform

Log transforms reduce right-skewed distributions, making them more Gaussian. This helps linear models and can improve tree-based models too.

In [ ]:
# Technique 1: Log Transform
df_fe = df.copy()

skewed_cols = ['income', 'total_charges', 'usage_minutes']
for col in skewed_cols:
    df_fe[f'{col}_log'] = np.log1p(df_fe[col].fillna(0))

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, skewed_cols):
    ax.hist(df[col].dropna(), bins=50, alpha=0.6, label='Original', color='#e74c3c', density=True)
    ax.hist(df_fe[f'{col}_log'].dropna(), bins=50, alpha=0.6, label='Log', color='#3498db', density=True)
    ax.set_title(f'{col}', fontsize=12, fontweight='bold')
    ax.legend()
plt.suptitle('Technique 1: Log Transform for Skewed Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

log_cols = num_cols + [f'{c}_log' for c in skewed_cols]
auc, std = evaluate_features(df_fe, log_cols)
improvements['+ Log Transform'] = auc
print(f'After log transform: ROC-AUC = {auc:.4f} (+/- {std:.4f}) | Baseline: {baseline_auc:.4f}')

### Technique 2: Power Transforms (Box-Cox / Yeo-Johnson)

Power transforms find the optimal power parameter to make the distribution as Gaussian as possible.

In [ ]:
# Technique 2: Power Transforms
pt = PowerTransformer(method='yeo-johnson')  # handles negative values
power_cols = ['income', 'total_charges', 'usage_minutes', 'days_since_last_activity']
valid_mask = df_fe[power_cols].notna().all(axis=1)
transformed = pt.fit_transform(df_fe.loc[valid_mask, power_cols].fillna(0))

for i, col in enumerate(power_cols):
    df_fe.loc[valid_mask, f'{col}_power'] = transformed[:, i]

print('Yeo-Johnson lambda parameters:')
for col, lam in zip(power_cols, pt.lambdas_):
    print(f'  {col}: lambda = {lam:.4f}')

power_feat_cols = num_cols + [f'{c}_power' for c in power_cols]
auc, std = evaluate_features(df_fe, power_feat_cols)
improvements['+ Power Transform'] = auc
print(f'\nAfter power transform: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

### Techniques 3-4: Scaling (StandardScaler + RobustScaler)

- **StandardScaler**: zero mean, unit variance. Best for Gaussian-like features.
- **RobustScaler**: uses median and IQR, resistant to outliers.

In [ ]:
# Techniques 3-4: Scaling
scaler_std = StandardScaler()
scaler_robust = RobustScaler()

for col in num_cols[:5]:
    vals = df_fe[col].fillna(df_fe[col].median()).values.reshape(-1, 1)
    df_fe[f'{col}_std'] = scaler_std.fit_transform(vals)
    df_fe[f'{col}_robust'] = scaler_robust.fit_transform(vals)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
col = 'income'
axes[0].hist(df_fe[col].dropna(), bins=40, color='#e74c3c', alpha=0.7)
axes[0].set_title('Original', fontweight='bold')
axes[1].hist(df_fe[f'{col}_std'].dropna(), bins=40, color='#3498db', alpha=0.7)
axes[1].set_title('StandardScaler', fontweight='bold')
axes[2].hist(df_fe[f'{col}_robust'].dropna(), bins=40, color='#2ecc71', alpha=0.7)
axes[2].set_title('RobustScaler', fontweight='bold')
plt.suptitle('Techniques 3-4: Scaling Comparison (Income)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Technique 5: Binning / Discretization

Convert continuous features into bins. Useful when the relationship with the target is non-linear or step-wise.

In [ ]:
# Technique 5: Binning
# Equal-width bins
df_fe['age_bin_equal'] = pd.cut(df_fe['age'], bins=5, labels=['very_young', 'young', 'middle', 'senior', 'elderly'])

# Quantile bins (equal frequency)
df_fe['age_bin_quantile'] = pd.qcut(df_fe['age'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

# Custom domain-knowledge bins
df_fe['income_bracket'] = pd.cut(df_fe['income'].fillna(0), 
                                  bins=[0, 30000, 60000, 100000, 200000, float('inf')],
                                  labels=['low', 'medium', 'high', 'very_high', 'ultra'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_fe.groupby('age_bin_equal')['target'].mean().plot(kind='bar', ax=axes[0], color='#3498db')
axes[0].set_title('Churn Rate by Age Bin (Equal Width)', fontweight='bold')
axes[0].set_ylabel('Churn Rate')
axes[0].tick_params(axis='x', rotation=45)

df_fe.groupby('income_bracket')['target'].mean().plot(kind='bar', ax=axes[1], color='#2ecc71')
axes[1].set_title('Churn Rate by Income Bracket', fontweight='bold')
axes[1].set_ylabel('Churn Rate')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### Techniques 6-7: Polynomial Features & Ratios

Create interaction and polynomial terms to capture non-linear relationships.

In [ ]:
# Technique 6: Polynomial features
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
poly_input = df_fe[['tenure_months', 'monthly_charges']].fillna(0)
poly_features = poly.fit_transform(poly_input)
poly_names = poly.get_feature_names_out(['tenure', 'charges'])

for i, name in enumerate(poly_names):
    df_fe[f'poly_{name}'] = poly_features[:, i]

print(f'Polynomial features created: {poly_names.tolist()}')

# Technique 7: Ratio features (domain knowledge)
df_fe['charge_per_month'] = df_fe['total_charges'] / (df_fe['tenure_months'] + 1)
df_fe['income_to_charges'] = df_fe['income'].fillna(0) / (df_fe['monthly_charges'] + 1)
df_fe['calls_per_month'] = df_fe['support_calls'] / (df_fe['tenure_months'] + 1)
df_fe['usage_per_charge'] = df_fe['usage_minutes'] / (df_fe['monthly_charges'] + 1)

ratio_cols = ['charge_per_month', 'income_to_charges', 'calls_per_month', 'usage_per_charge']
auc, std = evaluate_features(df_fe, num_cols + ratio_cols)
improvements['+ Ratios'] = auc
print(f'\nWith ratio features: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

### Techniques 8-10: Clipping, Rank Transform, Missing Value Indicators

Three simple but powerful techniques.

In [ ]:
# Technique 8: Clipping outliers
for col in ['income', 'total_charges']:
    q01, q99 = df_fe[col].quantile([0.01, 0.99])
    df_fe[f'{col}_clipped'] = df_fe[col].clip(q01, q99)

# Technique 9: Rank transform (percentile)
for col in ['income', 'usage_minutes']:
    df_fe[f'{col}_rank'] = df_fe[col].rank(pct=True)

# Technique 10: Missing value indicators
for col in ['income', 'satisfaction_score', 'feedback']:
    df_fe[f'{col}_missing'] = df_fe[col].isna().astype(int)

missing_cols = [c for c in df_fe.columns if c.endswith('_missing')]
print('Missing value indicator features:')
for col in missing_cols:
    pct = df_fe[col].mean() * 100
    target_rate = df_fe.loc[df_fe[col]==1, 'target'].mean() if df_fe[col].sum() > 0 else 0
    print(f'  {col}: {pct:.1f}% missing, churn rate when missing: {target_rate:.3f}')

extended_cols = num_cols + ratio_cols + missing_cols
auc, std = evaluate_features(df_fe, extended_cols)
improvements['+ Clip/Rank/Missing'] = auc
print(f'\nWith all numerical transforms: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

> **Key Takeaway -- Numerical Transforms:** Log transforms and ratio features are your highest-ROI numerical techniques. Always check feature skewness first (`df.skew()`), and create ratio features based on domain knowledge. Missing value indicators are free and often predictive -- missingness is rarely random.

<a id='3'></a>
## 3. 🏷️ Categorical Encoding (Techniques 11-20)

### Technique 11: Label Encoding

Simple integer encoding. Works well for ordinal features and tree-based models.

In [ ]:
# Technique 11: Label Encoding
cat_cols = ['gender', 'region', 'plan_type', 'payment_method', 'device_type']

for col in cat_cols:
    le = LabelEncoder()
    df_fe[f'{col}_label'] = le.fit_transform(df_fe[col].astype(str))

label_cols = [f'{c}_label' for c in cat_cols]
auc, std = evaluate_features(df_fe, num_cols + label_cols)
improvements['+ Label Encoding'] = auc
print(f'With label-encoded categoricals: ROC-AUC = {auc:.4f} (+/- {std:.4f})')
print(f'\nPlan type encoding: {dict(zip(df_fe["plan_type"].unique(), df_fe["plan_type_label"].unique()))}')

### Technique 12: One-Hot Encoding

Creates binary columns for each category. Best for nominal features with low cardinality.

In [ ]:
# Technique 12: One-Hot Encoding
ohe_cols = ['gender', 'plan_type', 'device_type']  # low cardinality only
df_ohe = pd.get_dummies(df_fe[ohe_cols], prefix=ohe_cols)
df_fe = pd.concat([df_fe, df_ohe], axis=1)

ohe_feature_cols = [c for c in df_ohe.columns]
auc, std = evaluate_features(df_fe, num_cols + ohe_feature_cols)
improvements['+ One-Hot'] = auc
print(f'One-hot encoded features: {ohe_feature_cols}')
print(f'With one-hot encoding: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

### Techniques 13-14: Frequency Encoding & Count Encoding

Replace categories with their frequency or count. Captures the rarity of each category.

In [ ]:
# Technique 13: Frequency Encoding
for col in cat_cols + ['city']:
    freq = df_fe[col].value_counts(normalize=True)
    df_fe[f'{col}_freq'] = df_fe[col].map(freq)

# Technique 14: Count Encoding
for col in cat_cols + ['city']:
    counts = df_fe[col].value_counts()
    df_fe[f'{col}_count'] = df_fe[col].map(counts)

freq_cols = [f'{c}_freq' for c in cat_cols + ['city']]
count_cols = [f'{c}_count' for c in cat_cols + ['city']]

auc, std = evaluate_features(df_fe, num_cols + freq_cols + count_cols)
improvements['+ Freq/Count Enc'] = auc
print(f'With frequency + count encoding: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

# Show city frequency distribution
fig, ax = plt.subplots(figsize=(12, 4))
df_fe['city_freq'].hist(bins=50, ax=ax, color='#3498db', edgecolor='white')
ax.set_title('Technique 13: City Frequency Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequency')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### Techniques 15-16: Ordinal Encoding & Binary Encoding

- **Ordinal**: encode with meaningful order (e.g., Basic < Standard < Premium)
- **Binary**: encode integers as binary digits

In [ ]:
# Technique 15: Ordinal Encoding (with domain knowledge)
plan_order = {'Basic': 0, 'Standard': 1, 'Premium': 2, 'Enterprise': 3}
df_fe['plan_ordinal'] = df_fe['plan_type'].map(plan_order)

# Technique 16: Binary Encoding (for medium-cardinality)
def binary_encode(series, col_name):
    """Encode labels as binary digits."""
    le = LabelEncoder()
    encoded = le.fit_transform(series.astype(str))
    n_bits = int(np.ceil(np.log2(len(le.classes_) + 1)))
    binary_cols = {}
    for bit in range(n_bits):
        binary_cols[f'{col_name}_bit{bit}'] = (encoded >> bit) & 1
    return pd.DataFrame(binary_cols)

binary_df = binary_encode(df_fe['region'], 'region')
df_fe = pd.concat([df_fe, binary_df], axis=1)

print(f'Region binary encoding ({binary_df.shape[1]} bits):')
print(binary_df.head())

### Techniques 17-20: Hash Encoding, Mean Encoding, Leave-One-Out Encoding, WoE

Advanced encoding techniques for high-cardinality features and target-informed encoding.

> **Key Takeaway -- Categorical Encoding:** For tree-based models, **target encoding** almost always outperforms one-hot and label encoding. But you MUST use K-fold cross-validation to prevent target leakage. For high-cardinality features (100+ categories), use frequency/hash encoding. For linear models, one-hot is still the safest choice.

In [ ]:
# Technique 17: Hash Encoding (for very high cardinality)
def hash_encode(series, n_components=8):
    """Hash encoding using Python's built-in hash."""
    result = {}
    for i in range(n_components):
        result[f'hash_{i}'] = series.apply(
            lambda x: int(abs(hash(str(x) + str(i))) % 1000) / 1000
        )
    return pd.DataFrame(result)

hash_df = hash_encode(df_fe['city'], n_components=8)
for col in hash_df.columns:
    df_fe[f'city_{col}'] = hash_df[col]

# Technique 18: Mean (Target) Encoding with smoothing
def target_encode_smooth(df, col, target, alpha=10):
    """Target encoding with Bayesian smoothing."""
    global_mean = df[target].mean()
    agg = df.groupby(col)[target].agg(['mean', 'count'])
    smooth = (agg['count'] * agg['mean'] + alpha * global_mean) / (agg['count'] + alpha)
    return df[col].map(smooth)

for col in cat_cols:
    df_fe[f'{col}_target_enc'] = target_encode_smooth(df_fe, col, 'target', alpha=10)

# Technique 19: Leave-One-Out Encoding
def loo_encode(df, col, target):
    """Leave-one-out target encoding."""
    target_sum = df.groupby(col)[target].transform('sum')
    target_count = df.groupby(col)[target].transform('count')
    return (target_sum - df[target]) / (target_count - 1)

df_fe['plan_loo'] = loo_encode(df_fe, 'plan_type', 'target')

# Technique 20: Weight of Evidence (WoE)
def woe_encode(df, col, target):
    """Weight of Evidence encoding."""
    events = df.groupby(col)[target].sum()
    non_events = df.groupby(col)[target].count() - events
    total_events = df[target].sum()
    total_non_events = len(df) - total_events
    
    dist_events = events / total_events
    dist_non_events = non_events / total_non_events
    
    woe = np.log((dist_non_events + 0.0001) / (dist_events + 0.0001))
    return df[col].map(woe)

for col in cat_cols:
    df_fe[f'{col}_woe'] = woe_encode(df_fe, col, 'target')

target_enc_cols = [f'{c}_target_enc' for c in cat_cols]
woe_cols = [f'{c}_woe' for c in cat_cols]

auc, std = evaluate_features(df_fe, num_cols + target_enc_cols)
improvements['+ Target Encoding'] = auc
print(f'With target encoding: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

auc_woe, _ = evaluate_features(df_fe, num_cols + woe_cols)
print(f'With WoE encoding:   ROC-AUC = {auc_woe:.4f}')

> **Key Takeaway -- DateTime Features:** DateTime columns are a goldmine of free signal. The most impactful features are: (1) **time-since-event** deltas, (2) **cyclical encodings** (sin/cos for month/day), and (3) **is_weekend/is_holiday** flags. Cyclical encoding prevents the model from treating December (12) as far from January (1).

<a id='4'></a>
## 4. 📅 DateTime Features (Techniques 21-27)

DateTime features are a goldmine of information. We extract temporal patterns at multiple granularities.

> **Key Takeaway -- Text Features:** Even simple text features (word count, sentiment word ratio, TF-IDF) can add significant signal to tabular models. For Kaggle competitions, a common winning pattern is to combine TF-IDF features with tree-based models, letting the model learn which text patterns matter.

In [ ]:
# Techniques 21-27: DateTime decomposition

# Technique 21: Basic components
df_fe['signup_year'] = df_fe['signup_date'].dt.year
df_fe['signup_month'] = df_fe['signup_date'].dt.month
df_fe['signup_day'] = df_fe['signup_date'].dt.day
df_fe['signup_dayofweek'] = df_fe['signup_date'].dt.dayofweek
df_fe['signup_quarter'] = df_fe['signup_date'].dt.quarter

# Technique 22: Is weekend/holiday
df_fe['signup_is_weekend'] = df_fe['signup_dayofweek'].isin([5, 6]).astype(int)
df_fe['signup_is_month_start'] = df_fe['signup_date'].dt.is_month_start.astype(int)
df_fe['signup_is_month_end'] = df_fe['signup_date'].dt.is_month_end.astype(int)

# Technique 23: Cyclical encoding (sin/cos for month, day of week)
df_fe['month_sin'] = np.sin(2 * np.pi * df_fe['signup_month'] / 12)
df_fe['month_cos'] = np.cos(2 * np.pi * df_fe['signup_month'] / 12)
df_fe['dow_sin'] = np.sin(2 * np.pi * df_fe['signup_dayofweek'] / 7)
df_fe['dow_cos'] = np.cos(2 * np.pi * df_fe['signup_dayofweek'] / 7)

# Technique 24: Time since events
ref_date = df_fe['signup_date'].max()
df_fe['days_since_signup'] = (ref_date - df_fe['signup_date']).dt.days
df_fe['days_since_login'] = (ref_date - df_fe['last_login']).dt.days

# Technique 25: Time between events
df_fe['signup_to_login_days'] = (df_fe['last_login'] - df_fe['signup_date']).dt.days

# Technique 26: Part of year
df_fe['signup_season'] = df_fe['signup_month'].map(
    {12: 'winter', 1: 'winter', 2: 'winter',
     3: 'spring', 4: 'spring', 5: 'spring',
     6: 'summer', 7: 'summer', 8: 'summer',
     9: 'fall', 10: 'fall', 11: 'fall'})

# Technique 27: Elapsed time ratios
df_fe['login_recency_ratio'] = df_fe['days_since_login'] / (df_fe['days_since_signup'] + 1)

# Visualize cyclical encoding
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
months = np.arange(1, 13)
axes[0].plot(months, np.sin(2 * np.pi * months / 12), 'o-', label='sin', color='#e74c3c')
axes[0].plot(months, np.cos(2 * np.pi * months / 12), 'o-', label='cos', color='#3498db')
axes[0].set_title('Technique 23: Cyclical Month Encoding', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].legend()

# Churn by signup month
df_fe.groupby('signup_month')['target'].mean().plot(kind='bar', ax=axes[1], color='#2ecc71')
axes[1].set_title('Churn Rate by Signup Month', fontweight='bold')
axes[1].set_ylabel('Churn Rate')
plt.tight_layout()
plt.show()

dt_cols = ['days_since_signup', 'days_since_login', 'signup_to_login_days', 
           'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'signup_is_weekend',
           'login_recency_ratio']
auc, std = evaluate_features(df_fe, num_cols + dt_cols)
improvements['+ DateTime'] = auc
print(f'With datetime features: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

> **Key Takeaway -- Interaction Features:** The most powerful interactions are **ratio features** (A/B) and **difference-from-group-mean** features. These capture "how does this customer compare to others in their group?" -- a question tree-based models struggle to answer from raw features alone.

<a id='5'></a>
## 5. 📝 Text Features (Techniques 28-34)

Extract numerical features from text data.

> **Key Takeaway -- Aggregation Features:** Group-based statistics (mean, std, rank within group) are among the **most impactful features in real competition solutions**. The pattern is: `df.groupby('category_col')['numeric_col'].transform('statistic')`. Z-scores within groups are especially powerful for detecting outliers relative to their peer group.

In [ ]:
# Fill missing feedback for text processing
df_fe['feedback_clean'] = df_fe['feedback'].fillna('')

# Technique 28: Text length features
df_fe['feedback_len'] = df_fe['feedback_clean'].str.len()
df_fe['feedback_word_count'] = df_fe['feedback_clean'].str.split().str.len().fillna(0)
df_fe['feedback_avg_word_len'] = df_fe['feedback_clean'].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if x.strip() else 0)

# Technique 29: Sentiment-like features (simple word counting)
positive_words = {'great', 'love', 'fast', 'reliable', 'amazing', 'wonderful', 
                  'excellent', 'fantastic', 'helpful', 'satisfied', 'happy', 'recommend', 'good'}
negative_words = {'terrible', 'slow', 'bad', 'disappointed', 'poor', 'worst', 
                  'regret', 'frustrating', 'complicated', 'confusing'}

df_fe['positive_count'] = df_fe['feedback_clean'].apply(
    lambda x: sum(1 for w in x.lower().split() if w in positive_words))
df_fe['negative_count'] = df_fe['feedback_clean'].apply(
    lambda x: sum(1 for w in x.lower().split() if w in negative_words))
df_fe['sentiment_ratio'] = (df_fe['positive_count'] + 1) / (df_fe['negative_count'] + 1)

# Technique 30: TF-IDF features
tfidf = TfidfVectorizer(max_features=20, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df_fe['feedback_clean'])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), 
                         columns=[f'tfidf_{w}' for w in tfidf.get_feature_names_out()])
df_fe = pd.concat([df_fe, tfidf_df], axis=1)

# Technique 31: Bag of Words count features
bow = CountVectorizer(max_features=10)
bow_matrix = bow.fit_transform(df_fe['feedback_clean'])
bow_df = pd.DataFrame(bow_matrix.toarray(),
                       columns=[f'bow_{w}' for w in bow.get_feature_names_out()])

# Technique 32: Character n-gram features
char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=10)
char_matrix = char_tfidf.fit_transform(df_fe['feedback_clean'])

# Technique 33: Special character counts
df_fe['feedback_exclamation'] = df_fe['feedback_clean'].str.count('!')
df_fe['feedback_question'] = df_fe['feedback_clean'].str.count(r'\?')
df_fe['feedback_caps_ratio'] = df_fe['feedback_clean'].apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1))

# Technique 34: Unique word ratio
df_fe['feedback_unique_ratio'] = df_fe['feedback_clean'].apply(
    lambda x: len(set(x.split())) / (len(x.split()) + 1) if x.strip() else 0)

text_cols = ['feedback_len', 'feedback_word_count', 'positive_count', 'negative_count',
             'sentiment_ratio', 'feedback_unique_ratio']
tfidf_cols = [c for c in df_fe.columns if c.startswith('tfidf_')]

auc, std = evaluate_features(df_fe, num_cols + text_cols + tfidf_cols)
improvements['+ Text Features'] = auc
print(f'With text features: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

# Visualize sentiment vs churn
fig, ax = plt.subplots(figsize=(10, 5))
for target_val, color, label in [(0, '#2ecc71', 'No Churn'), (1, '#e74c3c', 'Churn')]:
    mask = df_fe['target'] == target_val
    ax.hist(df_fe.loc[mask, 'sentiment_ratio'], bins=30, alpha=0.6, color=color, label=label)
ax.set_title('Technique 29: Sentiment Ratio Distribution by Churn', fontweight='bold', fontsize=13)
ax.set_xlabel('Sentiment Ratio (positive / negative word count)')
ax.legend()
plt.tight_layout()
plt.show()

<a id='6'></a>
## 6. 🔗 Interaction Features (Techniques 35-40)

Interaction features capture relationships between pairs of variables that individual features miss.

In [ ]:
# Technique 35: Multiplicative interactions
df_fe['tenure_x_charges'] = df_fe['tenure_months'] * df_fe['monthly_charges']
df_fe['calls_x_satisfaction'] = df_fe['support_calls'] * df_fe['satisfaction_score'].fillna(5)
df_fe['age_x_income'] = df_fe['age'] * df_fe['income'].fillna(0)

# Technique 36: Difference features
df_fe['charge_diff'] = df_fe['total_charges'] - df_fe['tenure_months'] * df_fe['monthly_charges']
df_fe['login_vs_activity'] = df_fe['days_since_login'] - df_fe['days_since_last_activity']

# Technique 37: Division with safety
df_fe['products_per_tenure'] = df_fe['num_products'] / (df_fe['tenure_months'] + 1)
df_fe['satisfaction_per_call'] = df_fe['satisfaction_score'].fillna(5) / (df_fe['support_calls'] + 1)

# Technique 38: Min/Max of feature pairs
df_fe['max_charge_metric'] = df_fe[['monthly_charges', 'total_charges']].apply(
    lambda row: max(row['monthly_charges'], row['total_charges'] / 100), axis=1)

# Technique 39: Categorical x Numerical interactions
for cat_col in ['plan_type', 'region']:
    for num_col in ['monthly_charges', 'tenure_months']:
        group_mean = df_fe.groupby(cat_col)[num_col].transform('mean')
        df_fe[f'{cat_col}_{num_col}_diff'] = df_fe[num_col] - group_mean
        df_fe[f'{cat_col}_{num_col}_ratio'] = df_fe[num_col] / (group_mean + 1)

# Technique 40: Combination of categorical features
df_fe['plan_x_device'] = df_fe['plan_type'].astype(str) + '_' + df_fe['device_type'].astype(str)
df_fe['plan_x_region'] = df_fe['plan_type'].astype(str) + '_' + df_fe['region'].astype(str)

# Encode combined categoricals
for col in ['plan_x_device', 'plan_x_region']:
    df_fe[f'{col}_freq'] = df_fe[col].map(df_fe[col].value_counts(normalize=True))

interaction_cols = ['tenure_x_charges', 'calls_x_satisfaction', 'charge_diff',
                    'products_per_tenure', 'satisfaction_per_call',
                    'plan_type_monthly_charges_diff', 'plan_type_tenure_months_diff',
                    'plan_x_device_freq', 'plan_x_region_freq']

auc, std = evaluate_features(df_fe, num_cols + interaction_cols)
improvements['+ Interactions'] = auc
print(f'With interaction features: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

<a id='7'></a>
## 7. 📊 Aggregation Features (Techniques 41-45)

Group-based statistics capture how each observation compares to its group.

In [ ]:
# Technique 41: Group statistics
for group_col in ['plan_type', 'region', 'device_type']:
    for agg_col in ['monthly_charges', 'tenure_months', 'support_calls']:
        for func in ['mean', 'std', 'median']:
            col_name = f'{group_col}_{agg_col}_{func}'
            df_fe[col_name] = df_fe.groupby(group_col)[agg_col].transform(func)

# Technique 42: Rank within group
for group_col in ['plan_type', 'region']:
    df_fe[f'income_rank_in_{group_col}'] = df_fe.groupby(group_col)['income'].rank(pct=True)
    df_fe[f'tenure_rank_in_{group_col}'] = df_fe.groupby(group_col)['tenure_months'].rank(pct=True)

# Technique 43: Deviation from group mean
for group_col in ['plan_type']:
    for num_col in ['monthly_charges', 'income']:
        group_mean = df_fe.groupby(group_col)[num_col].transform('mean')
        group_std = df_fe.groupby(group_col)[num_col].transform('std')
        df_fe[f'{num_col}_zscore_in_{group_col}'] = (df_fe[num_col] - group_mean) / (group_std + 1)

# Technique 44: Group size / proportion
for group_col in ['city', 'plan_type']:
    df_fe[f'{group_col}_group_size'] = df_fe.groupby(group_col)[group_col].transform('count')

# Technique 45: Cumulative features (simulating time-ordered data)
df_sorted = df_fe.sort_values('signup_date')
df_fe['cumulative_signups'] = df_sorted.groupby('plan_type').cumcount()
df_fe = df_fe.sort_index()  # restore original order

agg_cols = ([f'plan_type_{c}_{f}' for c in ['monthly_charges', 'tenure_months'] for f in ['mean', 'std']] +
            ['income_rank_in_plan_type', 'monthly_charges_zscore_in_plan_type',
             'city_group_size', 'plan_type_group_size'])

auc, std = evaluate_features(df_fe, num_cols + agg_cols)
improvements['+ Aggregations'] = auc
print(f'With aggregation features: ROC-AUC = {auc:.4f} (+/- {std:.4f})')

# Visualize group statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_fe.boxplot(column='monthly_charges', by='plan_type', ax=axes[0])
axes[0].set_title('Monthly Charges by Plan Type', fontweight='bold')
axes[0].set_xlabel('Plan Type')
plt.sca(axes[0])

df_fe.boxplot(column='monthly_charges_zscore_in_plan_type', by='plan_type', ax=axes[1])
axes[1].set_title('Z-Score Within Plan Type', fontweight='bold')
axes[1].set_xlabel('Plan Type')

fig.suptitle('Techniques 41-43: Group-Based Aggregation Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='8'></a>
## 8. 🎯 Target-Based Features (Techniques 46-50)

**Warning**: Target-based features must be computed with cross-validation to avoid leakage!

In [ ]:
<a id='10'></a>
## 10. Key Takeaways

### The 50 Techniques Summary

| # | Category | Techniques |
|---|----------|------------|
| 1-5 | Numerical | Log, Power, Standard Scale, Robust Scale, Binning |
| 6-10 | Numerical | Polynomial, Ratios, Clipping, Rank, Missing Indicators |
| 11-15 | Categorical | Label, One-Hot, Frequency, Count, Ordinal |
| 16-20 | Categorical | Binary, Hash, Mean/Target, Leave-One-Out, WoE |
| 21-27 | DateTime | Components, Weekend, Cyclical, Time-Since, Between, Season, Ratio |
| 28-34 | Text | Length, Sentiment, TF-IDF, BoW, Char n-grams, Special chars, Unique ratio |
| 35-40 | Interaction | Multiply, Difference, Division, Min/Max, Cat x Num, Cat x Cat |
| 41-45 | Aggregation | Group stats, Rank in group, Z-score, Group size, Cumulative |
| 46-50 | Target-Based | K-Fold target enc, Multi-stat, Count, Binned target, Crosses |

### Top Lessons from Competitions

1. **Target encoding is incredibly powerful** but must use proper K-fold cross-validation to prevent leakage.

2. **Ratio and interaction features** often outperform individual features because they capture relationships the model might not learn on its own.

3. **Group-based aggregations** (how does this observation compare to its group?) are among the most impactful features in real competition solutions.

4. **DateTime features are free signal** that many people overlook. Always extract cyclical encodings and time deltas.

5. **Missing value indicators** can be predictive features themselves, since missingness is often not random.

6. **Feature selection matters** -- more features is not always better. Use feature importance to prune.

7. **Domain knowledge beats generic techniques** -- the best features come from understanding the problem.

---

## Further Reading

**Kaggle Competition Write-ups (heavy feature engineering):**
- [1st Place - Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk/discussion/64821) -- masterclass in aggregation features
- [1st Place - IEEE Fraud Detection](https://www.kaggle.com/competitions/ieee-fraud-detection/discussion/111284) -- creative feature engineering
- [Top Solutions - American Express](https://www.kaggle.com/competitions/amex-default-prediction/discussion/348111) -- time-series aggregation features

**Libraries & Tools:**
- [Featuretools](https://www.featuretools.com/) -- automated feature engineering
- [Category Encoders](https://contrib.scikit-learn.org/category_encoders/) -- advanced categorical encoding library
- [Optuna](https://optuna.org/) -- hyperparameter optimization (including feature selection)

**Papers:**
- [A Survey on Feature Engineering](https://arxiv.org/abs/1904.12722) -- comprehensive academic survey
- [AutoFeat](https://arxiv.org/abs/1901.07329) -- automated feature engineering with selection

**Related Notebooks:**
- Check out my [RAG from Scratch](https://www.kaggle.com/lorenzoscaturchio) notebook for NLP/retrieval techniques
- Check out my [Attention Mechanisms Guide](https://www.kaggle.com/lorenzoscaturchio) for deep learning fundamentals

---

### **If this cookbook helped you, please upvote! Your support helps the Kaggle community discover useful resources.**

Next steps:
- Apply these techniques to a real competition dataset
- Experiment with automated feature engineering (Featuretools, AutoFeat)
- Combine with feature selection (Boruta, permutation importance)
- Try target encoding with different regularization strengths

**Connect with me** for more ML educational content!

<a id='9'></a>
## 9. 🚀 Full Pipeline: Combined Effect

Now let's combine the best features from each category and see the total improvement.

In [ ]:
# Combine best features from each category
best_features = (
    num_cols +  # Original numerical
    ratio_cols +  # Ratios
    missing_cols +  # Missing indicators
    target_enc_cols +  # Target encoding
    freq_cols +  # Frequency encoding
    dt_cols +  # DateTime
    text_cols +  # Text
    interaction_cols +  # Interactions
    agg_cols +  # Aggregations
    target_feat_cols  # Target-based
)

# Remove duplicates and ensure all columns exist
best_features = [c for c in dict.fromkeys(best_features) if c in df_fe.columns]

final_auc, final_std = evaluate_features(df_fe, best_features)
improvements['Full Pipeline'] = final_auc

print(f'Full Pipeline ROC-AUC: {final_auc:.4f} (+/- {final_std:.4f})')
print(f'Baseline ROC-AUC:      {baseline_auc:.4f}')
print(f'Improvement:           +{(final_auc - baseline_auc):.4f} ({(final_auc - baseline_auc) / baseline_auc * 100:.1f}%)')
print(f'\nTotal features used: {len(best_features)}')

In [ ]:
# Visualize the improvement journey
fig, ax = plt.subplots(figsize=(14, 6))

names = list(improvements.keys())
values = list(improvements.values())
colors_list = ['#95a5a6'] + ['#3498db'] * (len(names) - 2) + ['#e74c3c']

bars = ax.barh(names, values, color=colors_list, edgecolor='white', linewidth=2)

for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold', fontsize=10)

ax.set_xlabel('ROC-AUC', fontsize=12)
ax.set_title('Feature Engineering Impact: Cumulative ROC-AUC Improvement', 
             fontsize=14, fontweight='bold')
ax.axvline(x=baseline_auc, color='gray', linestyle='--', alpha=0.5)
ax.set_xlim(min(values) - 0.01, max(values) + 0.02)

plt.tight_layout()
plt.show()

### Feature Importance from Final Model

Which engineered features matter most?

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

X_final = df_fe[best_features].copy()
for col in X_final.columns:
    if X_final[col].dtype == 'object':
        X_final[col] = LabelEncoder().fit_transform(X_final[col].astype(str))
X_final = X_final.fillna(X_final.median())
y = df_fe['target']

model_final = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
model_final.fit(X_final, y)

importances = pd.Series(model_final.feature_importances_, index=best_features)
top_20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(12, 8))
top_20.sort_values().plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Top 20 Most Important Engineered Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Categorize top features
print('\nTop features by category:')
for feat in top_20.index:
    if feat in num_cols:
        cat = 'Original'
    elif 'target' in feat:
        cat = 'Target-Based'
    elif feat in ratio_cols:
        cat = 'Ratio'
    elif feat in dt_cols:
        cat = 'DateTime'
    elif feat in interaction_cols:
        cat = 'Interaction'
    elif feat in agg_cols:
        cat = 'Aggregation'
    else:
        cat = 'Other'
    print(f'  [{cat:>13}] {feat}: {importances[feat]:.4f}')

<a id='10'></a>
## 10. 📝 Key Takeaways

### The 50 Techniques Summary

| # | Category | Techniques |
|---|----------|------------|
| 1-5 | Numerical | Log, Power, Standard Scale, Robust Scale, Binning |
| 6-10 | Numerical | Polynomial, Ratios, Clipping, Rank, Missing Indicators |
| 11-15 | Categorical | Label, One-Hot, Frequency, Count, Ordinal |
| 16-20 | Categorical | Binary, Hash, Mean/Target, Leave-One-Out, WoE |
| 21-27 | DateTime | Components, Weekend, Cyclical, Time-Since, Between, Season, Ratio |
| 28-34 | Text | Length, Sentiment, TF-IDF, BoW, Char n-grams, Special chars, Unique ratio |
| 35-40 | Interaction | Multiply, Difference, Division, Min/Max, Cat x Num, Cat x Cat |
| 41-45 | Aggregation | Group stats, Rank in group, Z-score, Group size, Cumulative |
| 46-50 | Target-Based | K-Fold target enc, Multi-stat, Count, Binned target, Crosses |

### Top Lessons from Competitions

1. **Target encoding is incredibly powerful** but must use proper K-fold cross-validation to prevent leakage.

2. **Ratio and interaction features** often outperform individual features because they capture relationships the model might not learn on its own.

3. **Group-based aggregations** (how does this observation compare to its group?) are among the most impactful features in real competition solutions.

4. **DateTime features are free signal** that many people overlook. Always extract cyclical encodings and time deltas.

5. **Missing value indicators** can be predictive features themselves, since missingness is often not random.

6. **Feature selection matters** -- more features is not always better. Use feature importance to prune.

7. **Domain knowledge beats generic techniques** -- the best features come from understanding the problem.

---

**If this cookbook helped you, please upvote!** 🙏 It helps the Kaggle community discover useful resources.

Next steps:
- Apply these techniques to a real competition dataset
- Experiment with automated feature engineering (Featuretools, AutoFeat)
- Combine with feature selection (Boruta, permutation importance)
- Try target encoding with different regularization strengths

## Portfolio Quality Addendum

### Objective
Demonstrate which feature engineering patterns consistently improve tabular model performance.

### Data
Structured features with mixed numeric/categorical fields and realistic missingness patterns.

### Method
Apply transformations in controlled ablations to isolate uplift from each feature family.

### Evaluation
Compare cross-validation lift, stability, and feature importance drift across folds.

### Insight and Trade-off
- Insight: Simple interaction and aggregation features often outperform complex handcrafted pipelines.
- Because robust baseline transforms capture most non-linearity before heavy customization.
- Therefore standardize a high-signal feature baseline before niche transformations.
- Trade-off: aggressive feature expansion can improve peak score but reduce maintainability.
- Limitation: leakage risk rises when engineered features use future-aware aggregations.

## Conclusion and Next Steps

### Summary
A disciplined ablation workflow turns feature engineering from guesswork into repeatable gain.

### Next Steps
1. Add leakage checks for every time-dependent transformation.
2. Document per-feature computational cost versus score uplift.
3. Publish a reusable baseline feature module for new competitions.